# FENOMIND — Demo di Google Colab

Jalankan sel **dari atas ke bawah** (Shift+Enter). Tidak perlu GPU.

FENOMIND adalah alat **skrining awal**, bukan alat diagnosis.

In [ ]:
# 1. Unduh kode
import os
if not os.path.exists('/content/fenomind'):
    !git clone -q https://github.com/cindymariamsumitra-gif/fenomind /content/fenomind
os.chdir('/content/fenomind')
print('Folder kerja:', os.getcwd())

In [ ]:
# 2. Pasang pustaka (± 2-5 menit). Torch sudah tersedia di Colab.
!pip install -q -r requirements.txt
# Bila sel berikutnya error soal numpy/scikit-learn: menu Runtime -> Restart session, lalu jalankan lagi dari sel 1.

In [ ]:
# 3. Unduh model teks IndoBERT dari GitHub Releases (± 250 MB)
import os
if not os.path.exists('artifacts/text/indobert/final/model.safetensors'):
    !wget -q https://github.com/cindymariamsumitra-gif/fenomind/releases/download/V0.1/fenomind-indobert-final.zip -O /content/indobert.zip
    !unzip -q -o /content/indobert.zip -d /content/indobert_tmp
    !mkdir -p artifacts/text/indobert
    !cp -r /content/indobert_tmp/fenomind-ai/artifacts/text/indobert/final artifacts/text/indobert/
print('Model IndoBERT ada:', os.path.exists('artifacts/text/indobert/final/model.safetensors'))

In [ ]:
# 4. Muat FENOMIND
import json
from fenomind.api import inference as inf
from fenomind.visual.features import read_openface_csv

def tampilkan(hasil):
    print(json.dumps(hasil, indent=2, ensure_ascii=False)[:3000])

print('Model teks yang dipakai:', inf.text_backend())

In [ ]:
# 5. Skrining TEKS — ganti kalimatnya sesukamu
hasil = inf.screen_text('akhir-akhir ini aku merasa capek banget, tidak semangat kuliah, dan sering merasa sendirian')
print('Probabilitas :', hasil['probability'])
print('Interpretasi :', hasil['interpretation'])
print('Kata paling berpengaruh:', [w['word'] for w in hasil['explanation']['top_increasing'][:5]])
print('Keselamatan  :', hasil['safety'])
print('\n' + hasil['disclaimer'])

In [ ]:
# 6. Skrining VISUAL dari CSV OpenFace (contoh di folder examples/)
frames = read_openface_csv('examples/lmvd_155_label1.csv')
hasil = inf.screen_openface(frames)
print('Probabilitas :', hasil['probability'])
print('Kualitas     :', hasil['quality'])
print('Kelompok fitur paling berpengaruh:')
for f in hasil['explanation']['top_families']:
    print(f"  {f['contribution']:+.3f}  {f['label']}")
# Catatan: contoh ini termasuk data latih, jadi skornya hanya demo alur, bukan bukti performa.

In [ ]:
# 7. GABUNGAN (heuristik rata-rata, BELUM divalidasi)
v = inf.screen_openface(read_openface_csv('examples/lmvd_664_label0.csv'))
t = inf.screen_text('hari ini aku senang bisa jalan-jalan sama teman-teman ke pantai')
tampilkan(inf.fusion_block(v, t))

## (Opsional) Buka tampilan API interaktif (Swagger) di Colab
Jalankan sel di bawah, lalu klik tautan yang muncul.

In [ ]:
import threading, time, uvicorn
from fenomind.api.app import app
threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning'), daemon=True).start()
time.sleep(3)
from google.colab import output
output.serve_kernel_port_as_window(8000, path='/docs')